<hr>
<h1 style="text-align:center; color:#8B0000;">
LUNG PARENCHYMA SEGMENTATION DEMO
</h1>

<p style="text-align:center; font-size:16px;">
U-Net Based Medical Image Segmentation using Deep Learning
</p>

<hr>

***********************
# 1. INSTALLING REQUIRED LIBRARIES
*************

In [ ]:
!pip install gdown

******************
# 2. DOWNLOAD THE DATASET
******************

In [ ]:
import gdown

file_id = "17CzHmnOQhnK7D05aK6ZsGp3Jv11YOmEy" # The file ID where the lung parenchymna segmentation dataset is saved
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "lung_parenchyma_seg_dataset.rar", quiet=False)

************
# 3. EXTRACT DATASET
**********

In [ ]:
! unrar x lung_parenchyma_seg_dataset.rar

******************
# 4. IMPORT THE PYTHON LIBRARIES
***********

In [ ]:
import tqdm as tqdm
from tensorflow import keras
from tensorflow.keras import layers
import os, glob
import matplotlib.pyplot as plt
import cv2
import random
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import numpy as np
from keras.layers import Input, concatenate, Conv2D, MaxPooling2D, Conv2DTranspose, BatchNormalization, ReLU
from keras.callbacks import ModelCheckpoint, History
import tensorflow.keras.backend as K

*************
# 5. LOAD THE DATA PATHS AND SET THE PATH VARIABLES
***********

In [ ]:
TrainPath = r'/content/lung_parenchyma_seg_dataset/TRAIN'

ImagePath = os.path.join(TrainPath, 'IMAGES')
LabelPath = os.path.join(TrainPath, 'LABELS')

print(ImagePath, '\n', LabelPath)

In [ ]:
ValPath = r'/content/lung_parenchyma_seg_dataset/VAL'

ValImagePath = os.path.join(ValPath, 'IMAGES')
ValLabelPath = os.path.join(ValPath, 'LABELS')

print(ValImagePath, '\n', ValLabelPath)

**********
# 6. PREPROCESS AND PREPARE THE DATA IN THE FORM OF TENSOR
**********

<h2 style="color:#2E86C1;">Image Loading and Preprocessing Function</h2>

<p style="font-size:16px;">
Before training a deep learning model such as <b>U-Net</b>, the input CT images must be prepared in a structured and consistent format.
The <code>preprocess_images()</code> function performs a sequence of preprocessing operations that convert raw image files into
normalized tensors suitable for training.
</p>

<hr>

<h3 style="color:#117A65;">1. Parameter Initialization</h3>

<ul style="font-size:15px;">
<li><b>eps = 1e-7</b> : A very small constant added during normalization to prevent division-by-zero errors.</li>
<li><b>img_rows = 256</b> : Target height of the image.</li>
<li><b>img_columns = 256</b> : Target width of the image.</li>
</ul>

<p style="font-size:15px;">
These parameters ensure that every image in the dataset is resized to a consistent spatial resolution.
</p>

<hr>

<h3 style="color:#117A65;">2. Function Definition</h3>

<p style="font-size:15px;">
The function <code>preprocess_images(path)</code> accepts the directory path containing the input images and performs several preprocessing steps.
</p>

<ul style="font-size:15px;">
<li>Reads images from the specified directory</li>
<li>Converts images to grayscale</li>
<li>Resizes them to a fixed resolution</li>
<li>Normalizes pixel intensities</li>
<li>Converts them into a tensor suitable for neural network training</li>
</ul>

<hr>

<h3 style="color:#117A65;">3. Image File Collection</h3>

<p style="font-size:15px;">
All PNG images present in the directory are collected using the <code>glob</code> library.
</p>

<ul style="font-size:15px;">
<li><code>glob.glob(path + '/*.png')</code> retrieves all PNG files</li>
<li>The list is sorted to maintain consistent ordering</li>
<li>The total number of images is printed for verification</li>
</ul>

<hr>

<h3 style="color:#117A65;">4. Image Processing Loop</h3>

<p style="font-size:15px;">
Each image is processed individually inside a loop. The <code>tqdm</code> library is used to display a progress bar during processing.
</p>

<p style="font-size:15px;"><b>Processing steps performed on each image:</b></p>

<ol style="font-size:15px;">
<li>Read the image using <code>cv2.imread()</code></li>
<li>Convert the image from RGB to grayscale using <code>cv2.cvtColor()</code></li>
<li>Resize the image to <b>256 × 256</b> pixels</li>
<li>Convert the image to a floating-point NumPy array</li>
</ol>

<hr>

<h3 style="color:#117A65;">5. Image Normalization</h3>

<p style="font-size:15px;">
Pixel intensities are normalized to the range <b>[0,1]</b> using min-max normalization:
</p>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
Normalized Image = (Image - min(Image)) / (max(Image) - min(Image) + eps)
</pre>

<ul style="font-size:15px;">
<li>This improves numerical stability during training.</li>
<li>It ensures consistent intensity scaling across all images.</li>
</ul>

<hr>

<h3 style="color:#117A65;">6. Creating the Dataset Tensor</h3>

<p style="font-size:15px;">
All processed images are stored in a Python list and later converted into a NumPy array.
</p>

<ul style="font-size:15px;">
<li><code>np.asarray()</code> converts the list into a tensor</li>
<li><code>ImageArray[..., np.newaxis]</code> adds a channel dimension</li>
</ul>

<p style="font-size:15px;">
This converts the data into the shape:
</p>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
(Number of Images, Height, Width, Channels)
</pre>

<p style="font-size:15px;">
For grayscale images, the number of channels equals <b>1</b>.
</p>

<hr>

<h3 style="color:#117A65;">7. Final Output</h3>

<p style="font-size:15px;">
The function returns a 4-dimensional tensor containing all preprocessed images, which can directly be used as input to the neural network.
</p>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
Example Output Shape: (N, 256, 256, 1)
</pre>

<p style="font-size:15px;">
Here, <b>N</b> represents the number of images in the dataset.
</p>

In [ ]:
eps = 1e-7
img_rows = 256
img_columns = 256

def preprocess_images(path): #, temp = 5000
  '''preprocess functions take the images
    read them, resize them convert them to array
    normalize the images, and convert it to a trainable tensor'''
  ImageList=[]
  img_list=glob.glob(path + '/*.png')
  img_list.sort()
  #img_list = img_list[:temp]
  #print()
  print("No of images found:"+str(len(img_list)))
  for ii in tqdm.tqdm(range(len(img_list))):
    img = cv2.imread(img_list[ii])
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_img = cv2.resize(gray_img, (img_rows,img_columns))
    img_array = np.array(gray_img, dtype = np.float32)
    norm_im_array = (img_array - np.min(img_array))/((np.max(img_array) - np.min(img_array)+eps))
    ImageList.append(norm_im_array)
  ImageArray = np.asarray(ImageList, dtype = np.float32)
  ImageArray = ImageArray[..., np.newaxis]
  return ImageArray

In [ ]:
X_train = preprocess_images(ImagePath)
Y_train = preprocess_images(LabelPath)

X_val = preprocess_images(ValImagePath)
Y_val = preprocess_images(ValLabelPath)

*********
# 7. SANITY CHECK ON THE DATASET
***********

<h2 style="color:#2E86C1;">Sanity Checks and Data Verification</h2>

<p style="font-size:16px;">
Before training a deep learning model, it is essential to perform
<b>sanity checks</b> on the dataset. These checks help verify that the
images and corresponding segmentation masks are loaded correctly,
aligned properly, and stored in the expected tensor format.
</p>

<hr>

<h3 style="color:#117A65;">Why Sanity Checks Are Important</h3>

<ul style="font-size:15px;">
<li><b>Verify tensor dimensions</b> to ensure the dataset shape matches the expected input format for the neural network.</li>
<li><b>Confirm correct pairing of images and masks</b> so that each CT image corresponds to the correct segmentation mask.</li>
<li><b>Detect preprocessing errors</b> such as incorrect resizing, normalization issues, or channel mismatches.</li>
<li><b>Ensure visual correctness</b> by displaying image–mask pairs before model training.</li>
<li><b>Prevent training failures</b> that can occur due to incorrectly formatted data.</li>
</ul>

<hr>

<h3 style="color:#117A65;">Random Sample Visualization</h3>

<p style="font-size:15px;">
The following code snippet randomly selects an image from the training dataset
and visualizes both the <b>input CT image</b> and its corresponding
<b>segmentation mask</b>. This helps confirm that:
</p>

<ul style="font-size:15px;">
<li>The dataset has been loaded correctly.</li>
<li>The image and mask correspond to the same anatomical slice.</li>
<li>The segmentation mask correctly highlights the target region.</li>
</ul>

<hr>

<h3 style="color:#117A65;">Code Explanation</h3>

<ol style="font-size:15px;">

<li>
<b>Select a Random Training Sample</b><br>
A random index is generated using the <code>random.randint()</code> function.
This ensures that a different sample from the training dataset is visualized
each time the code is executed.
</li>

<br>

<li>
<b>Print the Selected Index</b><br>
Printing the index helps track which sample is being displayed, which can be
useful during debugging or dataset inspection.
</li>

<br>

<li>
<b>Create a Visualization Window</b><br>
A figure of size <b>10 × 10</b> is created to clearly display the image and
mask side by side.
</li>

<br>

<li>
<b>Display the Input Image</b><br>
The CT image is displayed using <code>matplotlib</code>. The
<code>np.squeeze()</code> function removes the extra channel dimension
(e.g., converting <code>(256,256,1)</code> into <code>(256,256)</code>)
so that the image can be properly visualized.
</li>

<br>

<li>
<b>Display the Corresponding Mask</b><br>
The segmentation mask is displayed next to the input image. This mask
represents the ground truth region that the neural network will learn to
segment.
</li>

<br>

<li>
<b>Remove Axis for Clean Visualization</b><br>
Axis labels are disabled to make the visualization cleaner and easier
to interpret.
</li>

In [ ]:
print(X_train.shape, '\t', type(X_train)) # NOTICE THE TENSOR SHAPE
print(X_val.shape, '\t', type(X_val))
print(Y_train.shape, '\t', type(Y_train))
print(Y_val.shape, '\t', type(Y_val))
print("Max. and Min. Intensities in Training dataset:", np.max(X_train), '\t', np.min(X_train))
print("Max. and Min. Intensities in Validation dataset:", np.max(X_val), '\t', np.min(X_val))

In [ ]:
# SANITY CHECK
X = random.randint(1, len(X_train)-1)
print(X)
plt.figure(figsize=(10, 10))
plt.subplot(121)
plt.imshow(np.squeeze(X_train[X,:,:]), cmap='gray')
plt.title('Image')
plt.axis('off')
plt.subplot(122)
plt.imshow(np.squeeze(Y_train[X,:,:]), cmap='gray')
plt.title('Mask')
plt.axis('off')
plt.show()

********************
# 8. CREATE THE U-NET SEGMENTATION MODEL
********************

<h2 style="color:#2E86C1;">U-Net Architecture for Lung Parenchyma Segmentation</h2>

<p style="font-size:16px;">
The following code implements a <b>U-Net convolutional neural network</b> using
TensorFlow/Keras. U-Net is a widely used architecture for
<b>medical image segmentation</b>, particularly effective in tasks such as
lung parenchyma segmentation from CT scans.
</p>

<p style="font-size:16px;">
The architecture follows a symmetric structure consisting of:
</p>

<ul style="font-size:15px;">
<li><b>Encoder (Contracting Path)</b> – extracts hierarchical features from the image.</li>
<li><b>Bottleneck</b> – captures the deepest and most abstract representation.</li>
<li><b>Decoder (Expanding Path)</b> – reconstructs the segmentation mask.</li>
<li><b>Skip Connections</b> – combine encoder and decoder features to preserve spatial information.</li>
</ul>

<hr>

<h3 style="color:#117A65;">1. Input Layer</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
inputs = Input(input_shape)
</pre>

<p style="font-size:15px;">
The input layer defines the shape of the images entering the network.
</p>

<ul style="font-size:15px;">
<li><b>256 × 256</b> – spatial dimensions of the CT slice</li>
<li><b>1</b> – grayscale channel</li>
</ul>

<p style="font-size:15px;">
Therefore, each training sample has the shape:
</p>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
(256, 256, 1)
</pre>

<hr>

<h3 style="color:#117A65;">2. Encoder (Feature Extraction)</h3>

<p style="font-size:15px;">
The encoder gradually reduces the spatial dimensions while increasing the
number of feature maps. This allows the network to capture increasingly
complex patterns from the input image.
</p>

<p style="font-size:15px;">
Each encoder block consists of:
</p>

<ul style="font-size:15px;">
<li><b>Conv2D</b> – extracts image features</li>
<li><b>ReLU activation</b> – introduces non-linearity</li>
<li><b>MaxPooling</b> – reduces spatial dimensions</li>
</ul>

<p style="font-size:15px;">
Example encoder block:
</p>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
c1 = Conv2D(8, (3,3), activation='relu', padding='same')(inputs)
p1 = MaxPooling2D((2,2))(c1)
</pre>

<p style="font-size:15px;">
After pooling, the spatial resolution is reduced:
</p>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
256 → 128 → 64 → 32 → 16 → 8
</pre>

<p style="font-size:15px;">
At the same time, the number of filters increases:
</p>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
8 → 16 → 32 → 64 → 128
</pre>

This enables the network to capture deeper features.

<hr>

<h3 style="color:#117A65;">3. Bottleneck Layer</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
c6 = Conv2D(256, (3,3), activation='relu', padding='same')(p5)
</pre>

<p style="font-size:15px;">
The bottleneck is the deepest part of the network.  
Here, the model learns high-level semantic features of the image
that help distinguish lung tissue from surrounding structures.
</p>

<p style="font-size:15px;">
This layer contains the largest number of filters (256),
allowing the network to learn rich representations.
</p>

<hr>

<h3 style="color:#117A65;">4. Decoder (Image Reconstruction)</h3>

<p style="font-size:15px;">
The decoder reconstructs the segmentation mask by gradually increasing
the spatial resolution of the feature maps.
</p>

<p style="font-size:15px;">
Each decoder block contains:
</p>

<ul style="font-size:15px;">
<li><b>Conv2DTranspose</b> – performs upsampling</li>
<li><b>Concatenation</b> – merges features from the encoder</li>
<li><b>Conv2D</b> – refines the combined features</li>
</ul>

Example decoder block:

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
u7 = Conv2DTranspose(128, (3,3), strides=(2,2), padding='same')(c6)
u7 = concatenate([u7, c5])
c7 = Conv2D(128, (3,3), activation='relu', padding='same')(u7)
</pre>

<hr>

<h3 style="color:#117A65;">5. Skip Connections</h3>

<p style="font-size:15px;">
A key feature of U-Net is the use of <b>skip connections</b>.
</p>

<p style="font-size:15px;">
These connections pass feature maps from the encoder directly
to the corresponding decoder layer:
</p>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
u7 = concatenate([u7, c5])
</pre>

<p style="font-size:15px;">
This helps the model:
</p>

<ul style="font-size:15px;">
<li>Preserve spatial details</li>
<li>Improve localization of structures</li>
<li>Generate sharper segmentation boundaries</li>
</ul>

<hr>

<h3 style="color:#117A65;">6. Output Layer</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
outputs = Conv2D(1, (1,1), activation='sigmoid')(c11)
</pre>

<p style="font-size:15px;">
The final layer produces the segmentation mask.
</p>

<ul style="font-size:15px;">
<li><b>1 filter</b> – because this is a binary segmentation problem</li>
<li><b>1×1 convolution</b> – combines features into a single probability map</li>
<li><b>Sigmoid activation</b> – outputs pixel-wise probabilities between 0 and 1</li>
</ul>

<p style="font-size:15px;">
Each pixel represents the probability that it belongs to the lung region.
</p>

<hr>

<h3 style="color:#117A65;">7. Model Creation</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
model = Model(inputs, outputs)
</pre>

<p style="font-size:15px;">
This line connects the input and output layers to create the complete
U-Net model.
</p>

<hr>

<h3 style="color:#117A65;">8. Model Summary</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
model.summary()
</pre>

<p style="font-size:15px;">
This command prints the full architecture of the network, including:
</p>

<ul style="font-size:15px;"> <b>Layer names</b> ; <b>Output tensor shapes</b> ; <b>Number of trainable parameters</b>
</ul>

<p style="font-size:15px;">
This helps verify that the architecture has been constructed correctly
before starting the training process.
</p>

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate
from tensorflow.keras.models import Model

def unet_model(input_shape=(256, 256, 1)):
    inputs = Input(input_shape)

    # Encoder
    c1 = Conv2D(8, (3, 3), activation='relu', padding='same')(inputs)
    p1 = MaxPooling2D((2, 2))(c1)  # 128

    c2 = Conv2D(16, (3, 3), activation='relu', padding='same')(p1)
    p2 = MaxPooling2D((2, 2))(c2)  # 64

    c3 = Conv2D(32, (3, 3), activation='relu', padding='same')(p2)
    p3 = MaxPooling2D((2, 2))(c3)  # 32

    c4 = Conv2D(64, (3, 3), activation='relu', padding='same')(p3)
    p4 = MaxPooling2D((2, 2))(c4)  # 16

    c5 = Conv2D(128, (3, 3), activation='relu', padding='same')(p4)
    p5 = MaxPooling2D((2, 2))(c5)  # 8

    # Bottleneck
    c6 = Conv2D(256, (3, 3), activation='relu', padding='same')(p5)

    # Decoder
    u7 = Conv2DTranspose(128, (3, 3), strides=(2, 2), padding='same')(c6)
    u7 = concatenate([u7, c5])
    c7 = Conv2D(128, (3, 3), activation='relu', padding='same')(u7)

    u8 = Conv2DTranspose(64, (3, 3), strides=(2, 2), padding='same')(c7)
    u8 = concatenate([u8, c4])
    c8 = Conv2D(64, (3, 3), activation='relu', padding='same')(u8)

    u9 = Conv2DTranspose(32, (3, 3), strides=(2, 2), padding='same')(c8)
    u9 = concatenate([u9, c3])
    c9 = Conv2D(32, (3, 3), activation='relu', padding='same')(u9)

    u10 = Conv2DTranspose(16, (3, 3), strides=(2, 2), padding='same')(c9)
    u10 = concatenate([u10, c2])
    c10 = Conv2D(16, (3, 3), activation='relu', padding='same')(u10)

    u11 = Conv2DTranspose(8, (3, 3), strides=(2, 2), padding='same')(c10)
    u11 = concatenate([u11, c1])
    c11 = Conv2D(8, (3, 3), activation='relu', padding='same')(u11)

    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c11)

    model = Model(inputs, outputs)
    return model


# Create model
model = unet_model()
model.summary()

****************
# 9. **CNN MODEL COMPILATION**
***********

<h2 style="color:#2E86C1;">Dice Coefficient and Dice Loss</h2>

<p style="font-size:16px;">
In medical image segmentation tasks such as <b>lung parenchyma segmentation</b>,
evaluating how well the predicted mask overlaps with the ground truth mask is very important.
One of the most widely used evaluation metrics for segmentation is the
<b>Dice Coefficient</b>
</p>

<p style="font-size:16px;">
The Dice coefficient measures the <b>overlap between two binary regions</b>:
</p>

<ul style="font-size:15px;">
<li>The <b>ground truth mask</b> (true segmentation) and The <b>predicted mask</b> generated by the neural network
</ul>

<hr>

<h3 style="color:#117A65;">Dice Coefficient Formula</h3>

<p style="font-size:15px;">
The Dice score is defined as:
</p>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
Dice = (2 × Intersection) / (Sum of pixels in prediction + Sum of pixels in ground truth)
</pre>

<p style="font-size:15px;">
The Dice value ranges between:
</p>

<ul style="font-size:15px;">
<li><b>0</b> → No overlap between prediction and ground truth</li>
<li><b>1</b> → Perfect overlap</li>
</ul>

<hr>

<h3 style="color:#117A65;">Smoothing Parameter</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
smooth = 1
eps = 1e-7
</pre>

<p style="font-size:15px;">
A small smoothing constant is added to the formula to avoid
<b>division by zero</b> when both masks contain no foreground pixels.
</p>

<ul style="font-size:15px;">
<li><b>smooth</b> stabilizes the Dice calculation.</li>
<li><b>eps</b> prevents numerical instability.</li>
</ul>

<hr>

<h3 style="color:#117A65;">Dice Coefficient Function</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
def dice_coef(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
</pre>

<p style="font-size:15px;">
The first step converts both the ground truth mask and predicted mask
into <b>one-dimensional vectors</b> using the <code>flatten()</code> function.  
This simplifies the calculation by treating the entire mask as a single vector.
</p>

<hr>

<h4 style="color:#7D3C98;">Intersection Calculation</h4>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
intersection = K.sum(y_true_f * y_pred_f)
</pre>

<p style="font-size:15px;">
The intersection represents the number of pixels that are correctly predicted
as belonging to the target region (e.g., lung tissue).
</p>

<ul style="font-size:15px;">
<li>If both prediction and ground truth contain a pixel → contributes to intersection</li>
<li>If only one contains the pixel → not counted</li>
</ul>

<hr>

<h4 style="color:#7D3C98;">Dice Score Computation</h4>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
return (2 * intersection + smooth) /
       (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)
</pre>

<p style="font-size:15px;">
The Dice coefficient doubles the intersection and divides it by the total
number of pixels in both masks. This emphasizes correct overlap between
prediction and ground truth.
</p>

<hr>

<h3 style="color:#117A65;">Dice Loss Function</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
def dice_coef_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)
</pre>

<p style="font-size:15px;">
Since neural networks are trained by <b>minimizing a loss function</b>,
the Dice coefficient is converted into a loss value.
</p>

<p style="font-size:15px;">
This is done by subtracting the Dice score from 1:
</p>

<ul style="font-size:15px;">
<li><b>Dice Score = 1</b> → Perfect segmentation → Loss = 0</li>
<li><b>Dice Score = 0</b> → No overlap → Loss = 1</li>
</ul>

<p style="font-size:15px;">
Thus, minimizing Dice loss during training forces the model
to maximize the overlap between predicted and ground truth masks.
</p>

In [ ]:
# DICE COEFFICIENT and DICE LOSS
smooth = 1
eps = 1e-7
def dice_coef(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * (intersection)+smooth ) / (K.sum(y_true_f) + K.sum(y_pred_f)+smooth)

def dice_coef_loss(y_true, y_pred):
    return 1-dice_coef(y_true, y_pred)

<h2 style="color:#2E86C1;">Model Compilation: Optimizer, Loss Function, and Metrics</h2>

<p style="font-size:16px;">
Before training a neural network, the model must be <b>compiled</b>.
Compilation defines three key components required for training:
</p>

<ul style="font-size:15px;">
<li><b>Optimizer</b> – determines how model weights are updated during training.</li>
<li><b>Loss Function</b> – measures how well the predicted output matches the ground truth.</li>
<li><b>Evaluation Metrics</b> – provides performance indicators during training.</li>
</ul>

<hr>

<h3 style="color:#117A65;">1. Optimizer</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
optimizer = keras.optimizers.Adam(0.0001)
</pre>

<p style="font-size:15px;">
The optimizer controls how the neural network updates its weights using
<b>gradient descent</b>.
</p>

<p style="font-size:15px;">
In this implementation, the <b>Adam optimizer</b> is used. Adam is one of the most
popular optimization algorithms for deep learning because it combines the
advantages of two methods:
</p>

<ul style="font-size:15px;">
<li><b>Momentum</b> – helps accelerate learning in relevant directions.</li>
<li><b>Adaptive learning rates</b> – adjusts step sizes automatically for each parameter.</li>
</ul>

<p style="font-size:15px;">
A smaller learning rate (~ 0.0001) ensures <b>stable and gradual learning</b>, which is
particularly important for medical image segmentation tasks.
</p>

<hr>

<h3 style="color:#117A65;">2. Loss Function</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
dice_loss = dice_coef_loss
</pre>

<p style="font-size:15px;">
The loss function measures how far the predicted segmentation mask is from
the ground truth mask.
</p>

<p style="font-size:15px;">
Here, the model uses <b>Dice Loss</b>, which is derived from the Dice coefficient.
Dice loss is particularly effective for segmentation problems because it
focuses on the <b>overlap between predicted and true regions</b>.
</p>

<ul style="font-size:15px;">
<li><b>Dice Score = 1</b> → perfect segmentation</li>
<li><b>Dice Score = 0</b> → no overlap</li>
</ul>

<p style="font-size:15px;">
Since neural networks minimize loss, the loss is defined as:
</p>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
Loss = 1 − Dice Score
</pre>

<p style="font-size:15px;">
Minimizing Dice loss therefore encourages the network to maximize the overlap
between predicted and ground truth masks.
</p>

<hr>

<h3 style="color:#117A65;">3. Evaluation Metric</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
metrics = [dice_coef]
</pre>

<p style="font-size:15px;">
While the loss function guides the learning process, evaluation metrics help
monitor model performance during training.
</p>

<p style="font-size:15px;">
In this case, the model tracks the <b>Dice coefficient</b>, which directly
measures segmentation accuracy based on the overlap between prediction
and ground truth masks.
</p>

<p style="font-size:15px;">
Tracking the Dice score during training allows us to see how well the model
is improving at identifying the lung region in CT images.
</p>

<hr>

<h3 style="color:#117A65;">4. Model Compilation</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
model.compile(optimizer, loss=dice_loss, metrics=metrics)
</pre>

<p style="font-size:15px;">
This step finalizes the configuration of the neural network before training begins.
</p>

<p style="font-size:15px;">
The model now knows:
</p>

<ul style="font-size:15px;">
<li>How to update its weights (<b>optimizer</b>)</li>
<li>How to measure prediction errors (<b>loss function</b>)</li>
<li>How to evaluate segmentation performance (<b>metrics</b>)</li>
</ul>

<p style="font-size:15px;">
Once the model is compiled, it is ready for the <b>training process</b>,
where it will learn to segment lung regions from CT images.
</p>

In [ ]:
# define optomizer
optimizer = keras.optimizers.Adam(0.0001)

# Define loss
dice_loss = dice_coef_loss 

# define metrics
metrics = [dice_coef]

#Compile the model

model.compile(optimizer, loss = dice_loss, metrics = metrics)

<h2 style="color:#2E86C1;">Checkpointing and Saving the Model</h2>

<p style="font-size:15px;">
During training, the model performance may fluctuate across epochs. 
To ensure that the <b>best-performing model</b> is preserved, we use the 
<b>ModelCheckpoint</b> callback from Keras. This automatically saves the model 
whenever the validation performance improves.
</p>

<hr>

<h3 style="color:#117A65;">Model Save Path</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
PATH = '/lung_seg_model.keras'
</pre>

<p style="font-size:15px;">
The model is saved in <code>.keras</code> format, which stores the complete model 
architecture and trained weights.
</p>

<hr>

<h3 style="color:#117A65;">Checkpoint Callback</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
model_checkpoint = [keras.callbacks.ModelCheckpoint(
    PATH,
    save_weights_only=False,
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)]
</pre>

<ul style="font-size:15px;">
<li><b>save_best_only=True</b> → saves only the best model</li>
<li><b>monitor='val_loss'</b> → evaluates performance on validation data</li>
<li><b>mode='min'</b> → lower validation loss indicates better performance</li>
</ul>

<p style="font-size:15px;">
This ensures that the final saved model corresponds to the 
<b>lowest validation loss achieved during training</b>.
</p>

In [ ]:
# CHECKPOINTING and SAVING

PATH = '/lung_seg_model.keras'

model_checkpoint = [keras.callbacks.ModelCheckpoint(PATH, 
                                                    save_weights_only=False, 
                                                    save_best_only=True, 
                                                    monitor='val_loss', 
                                                    mode='min')]

*********
# 10. TRAIN THE U-NET MODEL USING **model.fit()**
*********

<h2 style="color:#2E86C1;">Model Training</h2>

<p style="font-size:15px;">
The <code>model.fit()</code> function is used to train the U-Net segmentation model 
using the training dataset and evaluate its performance on validation data.
</p>

<hr>

<h3 style="color:#117A65;">Training Configuration</h3>

<pre style="background:#F4F6F7;padding:10px;border-radius:6px;">
history = model.fit(X_train,
                    Y_train,
                    batch_size=16,
                    epochs=15,
                    validation_data=(X_val, Y_val),
                    callbacks=[model_checkpoint])
</pre>

<ul style="font-size:15px;">
<li><b>X_train, Y_train</b> → training images and corresponding masks</li>
<li><b>batch_size = 16</b> → number of samples processed per iteration</li>
<li><b>epochs = 15</b> → number of times the model sees the entire dataset</li>
<li><b>validation_data</b> → evaluates performance on unseen validation images</li>
<li><b>callbacks</b> → saves the best model during training</li>
</ul>

<p style="font-size:15px;">
The training history is stored in <code>history</code>, which contains 
metrics and loss values for each epoch.
</p>

In [ ]:
history = model.fit(X_train,
                    Y_train,
                    batch_size=16,
                    epochs=15,
                    validation_data=(X_val, Y_val),
                    callbacks=[model_checkpoint],)

In [ ]:
# Plot training & validation iou_score values
plt.figure(figsize=(8,8))
plt.subplot(211)
plt.plot(history.history['dice_coef'])
plt.plot(history.history['val_dice_coef'])
plt.title('Model dice_score')
plt.ylabel('dice_score')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')

# Plot training & validation loss values
plt.subplot(212)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

*************
# 11. TEST THE MODEL'S PERFORMANCE ON THE TEST DATASET
***************

<h2 style="color:#2E86C1;">Model Prediction and Saving Results</h2>

<p style="font-size:15px;">
After training, the model is used to generate segmentation predictions on the 
<b>validation dataset</b>. The predicted masks are then saved for later 
visualization and evaluation.
</p>

<hr>

<h3 style="color:#117A65;">Prediction Function</h3>

<ul style="font-size:15px;">
<li><b>tf.keras.models.load_model()</b> load the saved model and model weights.</li>
<li><b>model.predict()</b> generates segmentation masks for the validation images.</li>
<li><b>verbose=1</b> shows the prediction progress.</li>
<li><b>predicted_mask.shape</b> verifies the output tensor dimensions.</li>
<li><b>np.save()</b> stores predictions as a NumPy file for later use.</li>
</ul>

In [ ]:
# Load the trained model
model = tf.keras.models.load_model('/lung_seg_model.keras',
                                    custom_objects={'dice_coef_loss': dice_coef_loss, 'dice_coef': dice_coef})


In [ ]:
def model_predict():
  print('Prediction Initialized')
  predicted_mask = model.predict(X_val, verbose=1)
  print('Predicted mask shape:', predicted_mask.shape)
  print('Prediction Done Successfully')

  np.save('/predictions.npy', predicted_mask)
  print('Predicted masks saved successfully')
    
model_predict()

In [ ]:
# LOAD THE PREDICTED IMAGES
predicted_mask = np.load('/predictions.npy')

In [ ]:
# VISUALIZE THE RESULTS
X = random.randint(0, 499)
print(f'Displaying Image No: {X}')
plt.figure(figsize=(20, 6))
plt.subplot(131)
plt.imshow(np.squeeze(X_val[X,:,:]), cmap='gray') # np.squeeze is used to eliminate the channel dimension
plt.title('Test_Image')
plt.axis('off')
plt.subplot(132)
plt.imshow(np.squeeze(Y_val[X,:,:]), cmap='gray')
plt.title('Ground_Truth')
plt.axis('off')
plt.subplot(133)
plt.imshow(np.squeeze(predicted_mask[X,:,:]), cmap='gray')
plt.title('Predicted_Mask')
plt.suptitle(f'Dice Coefficient: {dice_coef(Y_val[X,:,:], predicted_mask[X,:,:])}', fontsize=20)
plt.axis('off')
plt.show()

# Get The Extracted Lung Parenchyma

In [ ]:
lung_preserved_img = np.squeeze(X_val[X,:,:]) * np.squeeze(predicted_mask[X,:,:])

plt.figure(figsize=(8,8))
plt.imshow(lung_preserved_img, cmap='gray')
plt.title('Preserved Lung Parenchyma')
plt.axis('off')

*****************************
# 12.  IMPORT U-NET FROM GITHUB AND TRAIN THE NETWORK
****************************

In [ ]:
! pip install segmentation-models

In [ ]:
import tensorflow.keras as keras
import keras.utils

# Patch for compatibility with new keras
keras.utils.generic_utils = keras.utils

import segmentation_models as sm

sm.set_framework('tf.keras')
#print(sm.framework())

BACKBONE = 'vgg16'

model = sm.Unet(backbone_name = BACKBONE,
               input_shape = (256,256,1),
               classes = 1,
               activation = 'sigmoid',
               encoder_weights = None)

model.summary()

In [ ]:
# Learning Rate and Optimizer

LR = 0.0001 # You could experiment with different learning rates
optimizer = keras.optimizers.Adam(LR)

#Loss functions to optimize the network.

total_loss = sm.losses.DiceLoss() + (1 * sm.losses.BinaryCELoss()) 
print(f"Training with Dice + BinaryCrossEntropy Loss")

#Metrics to be monitored
metrics = [dice_coef,
           sm.metrics.IOUScore(threshold=0.5),]

# Compile
model.compile(optimizer, loss=total_loss, metrics=metrics)

print(f"Model compiled.")

In [ ]:
# Write your model.fit() function here and train the network also generate the results.